# GPU_ENVIRONMENT_CHECK

**Scientific question:** Can the locked SoilNet environment execute CUDA forward/backward safely?  
**Configuration:** `config/experiments/P0_final_soilnet.yaml`  
**Dataset/split:** hashes are verified, but no dataset loader is constructed.  
**Initialization:** synthetic SoilNet smoke with no checkpoint or pretrained download.  
**Expected outputs:** environment diagnostics only; no checkpoint or research metric.


## Environment and locked hashes

Resolve paths through ignored local config/environment variables and print all required versions and hashes.


In [1]:
from pathlib import Path
import json, sys

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the SoilNet repository")
sys.path.insert(0, str(REPO / "src"))

from soilnet.training import build_train_loader, build_val_loader, run_one_batch_preflight, train_experiment
from soilnet.utils import load_experiment_context, print_environment

CONFIG_PATH = REPO / "config/experiments/P0_final_soilnet.yaml"
RUN_TRAINING = False
RUN_GPU_SMOKE = True


In [2]:
import torch
from soilnet.models import SoilNetDualHead
from soilnet.training import compute_losses
from soilnet.utils import load_experiment_context, print_environment

context = load_experiment_context(CONFIG_PATH)
environment = print_environment(context)


python: 3.11.15
torch: 2.6.0+cu118
torchvision: 0.21.0+cu118
torchaudio: 2.6.0+cu118
timm: 1.0.29
cuda_available: True
cuda_device_count: 1
gpu: NVIDIA GeForce RTX 3050
torch_cuda_runtime: 11.8
seed: 20260905
config_sha256: 5bab9e5cf6610b59e2820ac813d579d9a590b5f70a50821908c18646960baf0c
manifest_sha256: 8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd
split_sha256: 8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f


## Conditional CUDA tensor and SoilNet smoke

No driver or CUDA package is installed here. If CUDA is unavailable, stop with `GPU_BLOCKED`.


In [3]:
if not torch.cuda.is_available():
    print("GPU_BLOCKED")
    print("Run scripts/gpu_diagnostic.py and check WSL GPU passthrough / host NVIDIA driver.")
elif RUN_GPU_SMOKE:
    device = torch.device("cuda")
    left = torch.ones((2, 2), device=device)
    print("matrix_multiplication:", left @ left)
    model = SoilNetDualHead(num_classes=10, use_light=True, backbone_pretrained=False).to(device)
    image = torch.randn(2, 3, 224, 224, device=device)
    light = torch.rand(2, 1, device=device)
    regression = torch.rand(2, 2, device=device)
    classification = torch.tensor([0, 1], device=device)
    prediction = model(image, light)
    losses = compute_losses(prediction[0], prediction[1], regression, classification, context.config)
    losses["total_loss"].backward()
    print({"GPU_SMOKE": "PASS", "forward": True, "backward": True, "research_metrics": False})


matrix_multiplication: tensor([[2., 2.],
        [2., 2.]], device='cuda:0')
{'GPU_SMOKE': 'PASS', 'forward': True, 'backward': True, 'research_metrics': False}


## Experiment Summary


In [4]:
print({"experiment_id": "GPU_ENVIRONMENT_CHECK", "training_completed": False,
       "checkpoint_path": None, "checkpoint_sha256": None,
       "final_training_metrics": None, "validation_metrics": None,
       "test_evaluated": "NO"})


{'experiment_id': 'GPU_ENVIRONMENT_CHECK', 'training_completed': False, 'checkpoint_path': None, 'checkpoint_sha256': None, 'final_training_metrics': None, 'validation_metrics': None, 'test_evaluated': 'NO'}
